# 05 — Prepare the panel forecasting dataset

This notebook performs dataset construction only. It converts the quality-filtered KEHA `panel_target` series from notebook 03 into chronological 1Q/2Q/4Q examples. It does not load, train, or evaluate an LLM; notebook 06 may start fine-tuning only if every gate below passes.

Training and validation examples are boundary-safe: their target quarter must remain inside their own split. Test examples require an available observed target.

In [ ]:
import hashlib, json, os
from pathlib import Path
import numpy as np
import pandas as pd
import yaml

def _find_repo():
    env = os.environ.get("JOBAI_REPO")
    if env:
        return Path(env).resolve()
    p = Path.cwd().resolve()
    for candidate in (p, *p.parents):
        if (candidate / "configs" / "eval.yaml").is_file():
            return candidate
    return p

REPO = _find_repo()
PRO = REPO / "data" / "processed"
MAN = REPO / "data" / "manifests"
REPORTS = REPO / "reports"
for directory in (PRO, MAN, REPORTS):
    directory.mkdir(parents=True, exist_ok=True)
CFG = yaml.safe_load((REPO / "configs" / "eval.yaml").read_text())
HORIZONS = [int(h) for h in CFG["horizons"]]
WINDOW = int(CFG["feature_window_quarters"])
SEED = int(CFG["seed"])
np.random.seed(SEED)
print("repo:", REPO)
print("horizons:", HORIZONS, "window:", WINDOW)

## Preconditions and selected panel series

In [ ]:
selection_path = PRO / "selected_series.csv"
assertions_path = PRO / "normalization_assertions.json"
baseline_manifest_path = REPORTS / "baseline_run_manifest.json"
assert selection_path.is_file(), "Run notebook 03 first"
assert assertions_path.is_file(), "Run notebook 02 first"
selection = pd.read_csv(selection_path)
panel_selection = selection[(selection["role"] == "panel_target") & selection["selected"].astype(bool)].copy()
assert not panel_selection.empty, "No panel target series passed notebook 03 selection"
normalization_assertions = json.loads(assertions_path.read_text())
assert all(result["status"] == "passed" for result in normalization_assertions.values())
print("selected panel series:", len(panel_selection))
print("baseline manifest present:", baseline_manifest_path.is_file())

## Construct boundary-safe examples

Each example stores the raw eight-quarter window and a scale-independent target. `target_log_change` is `log1p(target) - log1p(last observed value)`, allowing differently sized series to share one panel without treating their raw levels as comparable.

In [ ]:
splits = CFG["splits"]

def quarter_ordinal(q):
    return int(q[:4]) * 4 + int(q[-1]) - 1

def assigned_split(origin, target):
    if origin <= splits["train_end"] and target <= splits["train_end"]:
        return "train"
    if splits["val_start"] <= origin <= splits["val_end"] and target <= splits["val_end"]:
        return "validation"
    if splits["test_start"] <= origin <= splits["test_end"]:
        return "test"
    return None

tables = {}
for table_id in sorted(panel_selection["table_id"].unique()):
    tables[table_id] = pd.read_csv(PRO / f"{table_id}__normalized.csv", low_memory=False)

examples = []
skipped_missing_window = 0
skipped_missing_target = 0
for selected in panel_selection.itertuples(index=False):
    dimensions = json.loads(selected.dimensions_json)
    frame = tables[selected.table_id]
    for column, value in dimensions.items():
        frame = frame[frame[column].astype(str) == str(value)]
    frame = frame[["timeperiod_q", "value"]].copy()
    frame["value"] = pd.to_numeric(frame["value"], errors="coerce")
    frame["quarter_index"] = frame["timeperiod_q"].map(quarter_ordinal)
    frame = frame.sort_values("quarter_index").reset_index(drop=True)
    assert frame["timeperiod_q"].is_unique
    assert np.diff(frame["quarter_index"]).tolist() == [1] * (len(frame) - 1)
    quarters = frame["timeperiod_q"].tolist()
    values = frame["value"].to_numpy(dtype=float)
    for origin_idx in range(WINDOW - 1, len(values)):
        window_values = values[origin_idx - WINDOW + 1:origin_idx + 1]
        if not np.isfinite(window_values).all():
            skipped_missing_window += len(HORIZONS)
            continue
        origin = quarters[origin_idx]
        last_value = float(window_values[-1])
        window_mean = float(np.mean(window_values))
        window_std = float(np.std(window_values))
        scale = max(abs(last_value), window_std, 1.0)
        normalized_window = ((window_values - last_value) / scale).tolist()
        for horizon in HORIZONS:
            target_idx = origin_idx + horizon
            if target_idx >= len(values) or not np.isfinite(values[target_idx]):
                skipped_missing_target += 1
                continue
            target_quarter = quarters[target_idx]
            split = assigned_split(origin, target_quarter)
            if split is None:
                continue
            target_value = float(values[target_idx])
            identity = f"{selected.series_id}|{origin}|h{horizon}"
            examples.append({
                "example_id": hashlib.sha256(identity.encode()).hexdigest()[:20],
                "split": split, "table_id": selected.table_id, "series_family": selected.series_family,
                "series_id": selected.series_id, "dimensions_json": selected.dimensions_json,
                "origin_quarter": origin, "target_quarter": target_quarter, "horizon_q": horizon,
                "window_start_quarter": quarters[origin_idx - WINDOW + 1],
                "input_values_json": json.dumps([float(x) for x in window_values]),
                "normalized_input_json": json.dumps([float(x) for x in normalized_window]),
                "last_value": last_value, "window_mean": window_mean, "window_std": window_std, "scale": scale,
                "target_value": target_value,
                "target_scaled_change": (target_value - last_value) / scale,
                "target_log_change": float(np.log1p(target_value) - np.log1p(last_value)),
            })
dataset = pd.DataFrame(examples).sort_values(["split", "origin_quarter", "series_id", "horizon_q"]).reset_index(drop=True)
assert dataset["example_id"].is_unique
assert (dataset["target_quarter"].map(quarter_ordinal) > dataset["origin_quarter"].map(quarter_ordinal)).all()
assert (dataset.loc[dataset.split == "train", "target_quarter"] <= splits["train_end"]).all()
assert (dataset.loc[dataset.split == "validation", "target_quarter"] <= splits["val_end"]).all()
print("examples:", len(dataset))
print("skipped missing-window attempts:", skipped_missing_window)
print("skipped missing targets:", skipped_missing_target)

## Persist splits, summaries, and the fine-tuning gate

In [ ]:
combined_path = PRO / "panel_forecasting_dataset.parquet"
dataset.to_parquet(combined_path, index=False)
split_paths = {}
for split in ("train", "validation", "test"):
    split_frame = dataset[dataset["split"] == split].copy()
    path = PRO / f"panel_{split}.jsonl"
    split_frame.to_json(path, orient="records", lines=True, force_ascii=False)
    split_paths[split] = path

summary = (dataset.groupby(["split", "table_id", "series_family", "horizon_q"])
           .agg(n_examples=("example_id", "size"), n_series=("series_id", "nunique"),
                first_origin=("origin_quarter", "min"), last_origin=("origin_quarter", "max"),
                first_target=("target_quarter", "min"), last_target=("target_quarter", "max"))
           .reset_index())
summary_path = REPORTS / "panel_dataset_summary.csv"
summary.to_csv(summary_path, index=False)
display(summary)

gate_cfg = CFG["finetuning_gate"]
gate_checks = {
    "normalization_assertions_passed": all(result["status"] == "passed" for result in normalization_assertions.values()),
    "baseline_report_present": baseline_manifest_path.is_file(),
    "selected_panel_series": int(panel_selection["series_id"].nunique()),
    "minimum_selected_panel_series": int(gate_cfg["minimum_selected_panel_series"]),
    "train_examples": int((dataset["split"] == "train").sum()),
    "minimum_train_examples": int(gate_cfg["minimum_train_examples"]),
}
gate_checks["series_gate_passed"] = gate_checks["selected_panel_series"] >= gate_checks["minimum_selected_panel_series"]
gate_checks["example_gate_passed"] = gate_checks["train_examples"] >= gate_checks["minimum_train_examples"]
gate_checks["finetuning_ready"] = bool(gate_checks["normalization_assertions_passed"] and gate_checks["baseline_report_present"] and gate_checks["series_gate_passed"] and gate_checks["example_gate_passed"])
print("fine-tuning gate:", "PASS" if gate_checks["finetuning_ready"] else "FAIL")
print(json.dumps(gate_checks, indent=2))
if not gate_checks["finetuning_ready"]:
    print("decision: panel dataset may be used for analysis and classical experiments, but notebook 06 must not fine-tune an LLM")

In [ ]:
def file_record(path, rows):
    return {"path": str(path.relative_to(REPO)), "rows": int(rows), "sha256": hashlib.sha256(path.read_bytes()).hexdigest()}

manifest = {
    "name": "JobAI KEHA panel forecasting dataset",
    "responsibility": "dataset construction only; no model training",
    "feature_window_quarters": WINDOW, "horizons_q": HORIZONS, "splits": CFG["splits"],
    "selected_panel_series": int(panel_selection["series_id"].nunique()),
    "selection_catalog_sha256": hashlib.sha256(selection_path.read_bytes()).hexdigest(),
    "gate": gate_checks,
    "files": {"combined": file_record(combined_path, len(dataset))},
}
for split, path in split_paths.items():
    manifest["files"][split] = file_record(path, (dataset["split"] == split).sum())
manifest_path = MAN / "panel_dataset_card.json"
manifest_path.write_text(json.dumps(manifest, indent=2, ensure_ascii=False))
print("wrote:", combined_path)
for path in split_paths.values():
    print("wrote:", path)
print("wrote:", summary_path)
print("wrote:", manifest_path)

## Fine-tuning decision

Notebook 06 must read `panel_dataset_card.json` and stop before loading a base model when `gate.finetuning_ready` is false. A failed gate is a valid dataset result: the generated panel may still be used for analysis and classical forecasting experiments, but not for an LLM fine-tuning claim. The current selection must not be expanded, duplicated, or made less strict merely to turn the gate into a pass.